In [0]:
from datetime import datetime, timezone
import json
import requests
import time
import pandas as pd

default_date = "2026-08-28"  # Last Friday: 28 August 2026
# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "2. Schema Name")
dbutils.widgets.text("volume", "market_radar_landing", "3. Landing Volume")
dbutils.widgets.text("secret_scope", "valerii-matviiv-scope", "4. Secret Scope")
dbutils.widgets.text("secret_key", "finnhub-api-key", "5. Secret Key")
dbutils.widgets.text("tickers", "AAPL,NVDA,MSFT,AMZN,TSLA,QQQ", "6. Tickers")
dbutils.widgets.text("target_date", default_date, "7. Target Date (YYYY-MM-DD)")
dbutils.widgets.text("target_file_count", "500", "8. Target File Count")
dbutils.widgets.text("landing_override_path", "", "9. Override Landing Path")

# Retrieve widget values
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
secret_scope = dbutils.widgets.get("secret_scope")
secret_key = dbutils.widgets.get("secret_key")
tickers = [t.strip() for t in dbutils.widgets.get("tickers").split(",")]
target_date = dbutils.widgets.get("target_date").strip()
target_file_count = int(dbutils.widgets.get("target_file_count"))
override_path = dbutils.widgets.get("landing_override_path").strip()

# Resolve landing path
if override_path:
    landing_path = override_path
else:
    try:
        user_name = spark.sql("SELECT current_user()").collect()[0][0]
        if catalog == "workspace" or "gmail" in user_name:
            landing_path = f"/Workspace/Users/{user_name}/nasdaq_landing/landing/finnhub_news"
        else:
            landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/finnhub_news"
    except Exception:
        landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/finnhub_news"

dbutils.fs.mkdirs(landing_path)
print(f"Target Landing Path: {landing_path}")

# Retrieve API key if available
try:
    api_key = dbutils.secrets.get(scope=secret_scope, key=secret_key)
except Exception:
    api_key = ""

unique_articles = {}

# 1. Fetch General Market News
if api_key:
    try:
        general_url = f"https://finnhub.io/api/v1/news?category=general&token={api_key}"
        res_gen = requests.get(general_url, timeout=10)
        general_articles = res_gen.json() if res_gen.status_code == 200 and isinstance(res_gen.json(), list) else []
        for article in general_articles:
            art_id = str(article.get("id"))
            if art_id and art_id not in unique_articles:
                article["_schema_phase"] = "general_market"
                unique_articles[art_id] = article
        print(f"Collected {len(unique_articles)} general market articles.")
        time.sleep(1.1)

        # 2. Fetch Company News for Target Date
        for symbol in tickers:
            if len(unique_articles) >= target_file_count:
                break
            url = f"https://finnhub.io/api/v1/company-news?symbol={symbol}&from={target_date}&to={target_date}&token={api_key}"
            res = requests.get(url, timeout=10)
            if res.status_code == 200 and isinstance(res.json(), list):
                for article in res.json():
                    art_id = str(article.get("id"))
                    if art_id and art_id not in unique_articles:
                        article["_schema_phase"] = "nasdaq100_company"
                        article["index_tracker"] = "NASDAQ-100"
                        unique_articles[art_id] = article
                        if len(unique_articles) >= target_file_count:
                            break
            time.sleep(1.1)
    except Exception as e:
        print(f"Finnhub API fetch error: {e}")

# 3. Fallback Seed Data (guarantees landing data exists even without external API key in dev_free)
if not unique_articles:
    print("Generating realistic seed news dataset for target date...")
    seed_articles = [
        {"id": 91001, "datetime": int(pd.to_datetime(f"{target_date} 10:15:00").timestamp()), "headline": "Apple introduces next-gen M4 silicon with enhanced neural engine", "summary": "New benchmark results reveal unprecedented local AI processing efficiency for MacBook lineup.", "related": "AAPL", "source": "TechWire", "category": "company", "url": "https://example.com/aapl-m4"},
        {"id": 91002, "datetime": int(pd.to_datetime(f"{target_date} 10:45:00").timestamp()), "headline": "NVIDIA surges as cloud hyperscalers accelerate Blackwell deployment", "summary": "Enterprise datacenter capex forecasts revised upward following strong quarterly semiconductor shipment report.", "related": "NVDA", "source": "SemiconductorDaily", "category": "company", "url": "https://example.com/nvda-blackwell"},
        {"id": 91003, "datetime": int(pd.to_datetime(f"{target_date} 11:30:00").timestamp()), "headline": "Microsoft announces enterprise multi-agent Copilot workspace integration", "summary": "Office suite productivity upgrades drive double-digit enterprise subscription expansion.", "related": "MSFT", "source": "CloudInsider", "category": "company", "url": "https://example.com/msft-copilot"},
        {"id": 91004, "datetime": int(pd.to_datetime(f"{target_date} 12:15:00").timestamp()), "headline": "Amazon AWS rolls out custom Graviton instances to reduce enterprise hosting costs", "summary": "Cloud margins expand as next-generation server clusters achieve higher power efficiency.", "related": "AMZN", "source": "RetailTech", "category": "company", "url": "https://example.com/amzn-graviton"},
        {"id": 91005, "datetime": int(pd.to_datetime(f"{target_date} 13:00:00").timestamp()), "headline": "Tesla expands full self-driving hardware production for robotaxi fleet rollout", "summary": "Autonomous mileage milestones surpassed across North American commercial pilot corridors.", "related": "TSLA", "source": "EVWeekly", "category": "company", "url": "https://example.com/tsla-robotaxi"},
        {"id": 91006, "datetime": int(pd.to_datetime(f"{target_date} 19:30:00").timestamp()), "headline": "Nasdaq 100 Index rallies amid sustained semiconductor momentum", "summary": "Tech heavyweights lead benchmark index to fresh record highs in late afternoon trading.", "related": "QQQ", "source": "MarketWatch", "category": "general", "url": "https://example.com/qqq-rally"}
    ]
    for a in seed_articles:
        unique_articles[str(a["id"])] = a

# 4. Write each unique article as an individual landing JSON file keyed by ArticleId
all_articles = list(unique_articles.values())[:target_file_count]
for article in all_articles:
    art_id = str(article.get("id"))
    article["_ingested_at"] = datetime.now(timezone.utc).isoformat()
    file_name = f"{landing_path}/news_{art_id}.json"
    with open(file_name, "w") as f:
        json.dump(article, f)

print(f"Successfully landed {len(all_articles)} unique news files into: {landing_path}")


In [0]:
# catalog = dbutils.widgets.get("catalog")
# schema = dbutils.widgets.get("schema")
# volume = dbutils.widgets.get("volume")
# landing_path = f"/Volumes/{catalog}/{schema}/{volume}/landing/finnhub_news"

# files = dbutils.fs.ls(landing_path)
# print(f"Total Landed JSON Files: {len(files)}")

# if files:
#     print("--- Sample Landed JSON Content ---")
#     sample_path = files[0].path
#     sample_json = spark.read.option("multiline", "true").json(sample_path)
#     display(sample_json)